In [1]:
from pyomo.environ import *
from pyomo.gdp import Disjunct, Disjunction

In [2]:
model = ConcreteModel()

In [3]:
products = ["P", "Q"]
reactors = [1, 2]

In [4]:
# Data, one entry per (reactor, product)
model.profit = Param(reactors, products, initialize={
    (1, "P"): 50, (1, "Q"): 40,
    (2, "P"): 70, (2, "Q"): 40,
})
model.steam = Param(reactors, products, initialize={
    (1, "P"): 2, (1, "Q"): 1,
    (2, "P"): 3, (2, "Q"): 1,
})

model.feedLimit = Param(initialize=60)
model.demandLimit = Param(initialize=35)
model.steamLimit = Param(initialize=90)

In [5]:
model.make = Var(products, within=NonNegativeReals, bounds=(0, 60))
model.Profit = Var(within=Reals, bounds=(0, 4200))

In [6]:
model.MaximizeProfit = Objective(expr = model.Profit, sense = maximize)

In [7]:
model.FeedConstraint = Constraint(expr = sum(model.make[i] for i in products) <= model.feedLimit)
model.DemandConstraint = Constraint(expr = model.make["P"] <= model.demandLimit)

In [8]:
# One Disjunct per reactor, filled by a loop
model.R = Disjunct(reactors)

for r in reactors:
    model.R[r].ProfitConstraint = Constraint(
        expr = model.Profit == sum(model.profit[r, i]*model.make[i] for i in products))
    model.R[r].SteamConstraint = Constraint(
        expr = sum(model.steam[r, i]*model.make[i] for i in products) <= model.steamLimit)

In [9]:
model.reactorChoice = Disjunction(expr = [model.R[r] for r in reactors], xor=True)

In [10]:
TransformationFactory("gdp.bigm").apply_to(model)

In [11]:
optimizer = SolverFactory("appsi_highs")
optimizer.solve(model)
model.display()

Model unknown

  Variables:
    make : Size=2, Index={P, Q}
        Key : Lower : Value : Upper : Fixed : Stale : Domain
          P :     0 :  15.0 :    60 : False : False : NonNegativeReals
          Q :     0 :  45.0 :    60 : False : False : NonNegativeReals
    Profit : Size=1, Index=None
        Key  : Lower : Value              : Upper : Fixed : Stale : Domain
        None :     0 : 2850.0000000000005 :  4200 : False : False :  Reals

  Objectives:
    MaximizeProfit : Size=1, Index=None, Active=True
        Key  : Active : Value
        None :   True : 2850.0000000000005

  Constraints:
    FeedConstraint : Size=1
        Key  : Lower : Body : Upper
        None :  None : 60.0 :  60.0
    DemandConstraint : Size=1
        Key  : Lower : Body : Upper
        None :  None : 15.0 :  35.0

  Blocks:
    Block _pyomo_gdp_bigm_reformulation
    
      Variables:
        None
    
      Objectives:
        None
    
      Constraints:
        reactorChoice_xor : Size=1
            Key

In [12]:
print(value(model.R[1].indicator_var), value(model.R[2].indicator_var))

False True
